# 🌐 Module 1: 6G Multi-MNO Synthetic Data Generation & Database Management

## 📌 1. Project Overview & Context
In 6G Wireless Communication Networks, **Dynamic Spectrum Sharing (DSS)** among multiple Mobile Network Operators (MNOs) is critical for maximizing spectral efficiency and avoiding interference with Primary Users (PUs). 

This notebook simulates a realistic **6G Multi-MNO Radio Frequency (RF) Environment** and manages the storage of spectrum observation logs and MNO traffic demand in a structured database.

---

## 📐 2. Mathematical & Algorithmic Background

### A. Primary User (PU) Channel Occupancy (Markov Chain)
The PU occupancy state $S_{t, c} \in \{0, 1\}$ for step $t$ and channel $c$ is modeled using a 2-state Discrete Time Markov Chain (DTMC):
- **State 0 (Idle/Free)**: Channel available for Secondary User (MNO) opportunistic access.
- **State 1 (Busy/Occupied)**: PU is transmitting; MNO must not interfere.

Transition Probabilities:
$$P(S_{t+1}=1 \mid S_t=0) = \lambda_{ON}$$
$$P(S_{t+1}=0 \mid S_t=1) = \mu_{OFF}$$

### B. Radio Frequency & Fading Model
The received signal power $P_{rx}$ at channel $c$ under Rayleigh fading and additive thermal noise is:
$$P_{rx}(dBm) = P_{tx} - L_{path} + 10\log_{10}\left(|h|^2\right) + \eta_t$$
where $h \sim \mathcal{CN}(0, 1)$ represents Rayleigh fading channel gain, and $\eta_t$ represents noise & interference fluctuations.

The Signal-to-Noise Ratio (SNR) is calculated as:
$$\text{SNR}_{est} (dB) = P_{rx} - \text{Noise}_{est}$$

### C. Secondary User (MNO) Traffic Demand Model
For $M$ competing MNOs, active user count $U_{m, t}$ follows a diurnal (time-of-day) sinusoidal demand curve with Poisson noise spikes:
$$U_{m, t} = U_{peak} \cdot \left[ 0.3 + 0.7 \sin^2\left(\frac{\pi \cdot t}{T_{day}}\right) \right] + \text{Poisson}(\lambda_{spike})$$
The requested bandwidth in MHz is proportional to active users and target spectral efficiency $SE_{ref}$.

---

## 🗄️ 3. Database Schema & Architecture
We store generated datasets in **SQLite** (or PostgreSQL via `.env` connection string):
1. **`channel_obs` Table**: Stores time-series spectrum sensing observations across 16 RF channels.
2. **`mno_demand` Table**: Stores active user demand logs across 4 MNOs.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dss.data_generation.simulator import generate_dataset
from dss.data_generation.scenarios import list_scenarios
from dss.database.connection import get_connection, get_db_url

print("✅ Environment setup complete.")
print("Available Scenarios:", list_scenarios())
print("Database URL:", get_db_url())

ModuleNotFoundError: No module named 'matplotlib'

## ⚡ 4. Synthetic Data Generation across Scenarios
We generate synthetic data for the scenario `normal` spanning 14 days with 5-minute sampling intervals (4,032 time steps).

In [ ]:
# Generate synthetic dataset for normal scenario
scenario = "normal"
seed = 42

channel_obs_df, mno_demand_df, dataset_key = generate_dataset(
    scenario_name=scenario,
    seed=seed,
    save=True
)

print(f"🎉 Dataset Generation Successful!")
print(f"Dataset Key: {dataset_key}")
print(f"ChannelObs Shape: {channel_obs_df.shape} (Steps x Channels = 4032 x 16)")
print(f"MnoDemand Shape: {mno_demand_df.shape} (Steps x MNOs = 4032 x 4)")

# Display first few rows of ChannelObs
channel_obs_df.head()

In [ ]:
# Display first few rows of MnoDemand
mno_demand_df.head()

## 💾 5. Database Table Creation & Data Ingestion
Now we connect to SQLite database (`data/dss_database.db`), create SQL table schemas, and bulk insert our generated simulation data.

In [ ]:
conn = get_connection()
cursor = conn.cursor()

# 1. Create channel_obs table schema
cursor.execute("""
CREATE TABLE IF NOT EXISTS channel_obs (
    step_idx INTEGER NOT NULL,
    ts TEXT NOT NULL,
    channel_id INTEGER NOT NULL,
    pu_state INTEGER NOT NULL,
    noise_true_dbm REAL NOT NULL,
    noise_est_dbm REAL NOT NULL,
    interference_dbm REAL NOT NULL,
    rx_power_dbm REAL NOT NULL,
    snr_est_db REAL NOT NULL,
    PRIMARY KEY (step_idx, channel_id)
);
""")

# 2. Create mno_demand table schema
cursor.execute("""
CREATE TABLE IF NOT EXISTS mno_demand (
    step_idx INTEGER NOT NULL,
    ts TEXT NOT NULL,
    mno_id INTEGER NOT NULL,
    active_users INTEGER NOT NULL,
    traffic_demand_mbps REAL NOT NULL,
    requested_bw_mhz REAL NOT NULL,
    PRIMARY KEY (step_idx, mno_id)
);
""")

conn.commit()
print("✅ SQL Tables Created Successfully.")

# Save DataFrames into SQL tables
channel_obs_df.to_sql("channel_obs", conn, if_exists="replace", index=False)
mno_demand_df.to_sql("mno_demand", conn, if_exists="replace", index=False)
print("📥 Simulation Data Successfully Ingested into Database!")

## 📊 6. Database Analytical Queries & Summary Statistics
We can now run SQL queries to aggregate spectrum metrics directly from our database.

In [ ]:
# Query 1: Channel Occupancy & Average SNR per RF Channel
query1 = """
SELECT 
    channel_id,
    COUNT(*) as total_steps,
    SUM(pu_state) as busy_steps,
    ROUND(AVG(pu_state) * 100, 2) as occupancy_rate_pct,
    ROUND(AVG(snr_est_db), 2) as avg_snr_db,
    ROUND(AVG(rx_power_dbm), 2) as avg_rx_power_dbm
FROM channel_obs
GROUP BY channel_id
ORDER BY channel_id;
"""
channel_stats = pd.read_sql_query(query1, conn)
channel_stats

In [ ]:
# Query 2: MNO Traffic Demand Aggregations
query2 = """
SELECT 
    mno_id,
    AVG(active_users) as avg_users,
    MAX(active_users) as max_users,
    ROUND(AVG(traffic_demand_mbps), 2) as avg_traffic_mbps,
    ROUND(AVG(requested_bw_mhz), 2) as avg_requested_bw_mhz
FROM mno_demand
GROUP BY mno_id;
"""
mno_stats = pd.read_sql_query(query2, conn)
mno_stats

## 📉 7. Data Visualization & Insights
Let's visualize the Primary User (PU) occupancy heatmap and MNO traffic demand curves over time.

In [ ]:
plt.figure(figsize=(14, 5))

# Pivot table for channel observations (first 288 steps = 24 hours)
pivot_rx = channel_obs_df[channel_obs_df["step_idx"] < 288].pivot(
    index="channel_id", columns="step_idx", values="rx_power_dbm"
)

sns.heatmap(pivot_rx, cmap="viridis", cbar_kws={'label': 'Received Power (dBm)'})
plt.title("6G Multi-Channel Received Power Spectrogram (First 24 Hours)", fontsize=14, fontweight="bold")
plt.xlabel("Time Step (5-min intervals)")
plt.ylabel("RF Channel ID (0 to 15)")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
for mno in range(4):
    sub_df = mno_demand_df[(mno_demand_df["mno_id"] == mno) & (mno_demand_df["step_idx"] < 288)]
    plt.plot(sub_df["step_idx"], sub_df["traffic_demand_mbps"], label=f"MNO {mno}")

plt.title("MNO Diurnal Traffic Demand over 24 Hours", fontsize=14, fontweight="bold")
plt.xlabel("Time Step (5-min intervals)")
plt.ylabel("Traffic Demand (Mbps)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 🎯 Summary & Next Steps
- Synthetic data for 16 6G RF channels and 4 MNOs successfully generated.
- Database schema initialized and data populated into SQLite (`dss_database.db`).
- Analytical queries and spectrogram visualizations verify realistic RF behavior.
- **Next Notebook**: `02_Spectrum_Sensing_VAE_GMM.ipynb` (Spectrum occupancy sensing using PyTorch VAE + GMM).